# Test: `ClusterPicker`

This notebook imports `cluster_picker_1a.py` directly and uses the built-in `plot_` methods for the charts. Each part ends with chart checks that call every `plot_` method.

This notebook uses two well-known datasets:

| Part | Dataset | Why it's a good test |
|---|---|---|
| **A** | **Mall Customers** (200 shoppers) | The classic customer-segmentation example. Annual income vs spending score has **5 visible groups**, so we know what the picker should find |
| **B** | **Palmer Penguins** (333 penguins) | We hide the species and see if clustering finds them. Passing `ground_truth="species"` lets us **score the clusters against reality**, which shows where the internal metrics mislead |

Each part ends with automatic checks that print **All checks passed**.

In [ ]:
# --- Find the ML root folder and import the picker ---
import sys
from pathlib import Path

ml_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pickers").is_dir())
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pickers.cluster_picker_1a import ClusterPicker
print("Using:", ml_root / "pickers" / "cluster_picker_1a.py")

# Part A: Mall Customers
## A1. Load the data

In [ ]:
MALL_URL = "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv"
mall = pd.read_csv(MALL_URL)
print(mall.shape, "| missing values:", mall.isna().sum().sum())
mall.head()

## A2. Set up the picker
We cluster on **income and spending only**, the classic version of this exercise. `ignore=` leaves the other columns out of the clustering but keeps them in the DataFrame, so they still appear in `assign_clusters()`.

Creating the picker runs a K-Means scan of k = 2 to 10 to choose the starting number of clusters.

In [ ]:
mall_cl = ClusterPicker(mall, ignore=["CustomerID", "Gender", "Age"])
print(mall_cl.summary())
mall_cl.k_scan()

In [ ]:
# The k-scan as charts: best k marked on each panel, the starting k as a dotted line
mall_cl.plot_k_scan();

## A3. Compare all models

In [ ]:
%%time
mall_lb = mall_cl.compare_models()
mall_lb

In [ ]:
mall_cl.plot_leaderboard();

## A4. Tune the top 3

In [ ]:
mall_tuned = mall_cl.tune_top(n=3, n_iter=20)
print(mall_cl.summary())
mall_tuned

In [ ]:
mall_cl.plot_tuning();

## A5. Profile the segments
This is the table you'd hand to a marketing manager: the size and average values of each cluster, in the original units.

In [ ]:
best_mall = mall_cl.best_model_id()
mall_cl.profile(best_mall)

In [ ]:
mall_cl.plot_profile();          # colour = above/below average, numbers = actual averages

With only two numeric columns, `plot_clusters()` uses the original axes. With more, it switches to a 2-D PCA view.

In [ ]:
seg = mall_cl.assign_clusters(best_mall)
mall_cl.plot_clusters();

In [ ]:
mall_cl.plot_silhouette();       # bars below 0 = points that may be in the wrong cluster

For comparison, DBSCAN finds its own number of clusters and marks outliers as **noise** (grey x).

In [ ]:
mall_cl.plot_clusters("dbscan");

## A6. Label new customers
Centroid models like K-Means can `.predict()` new rows. Density and hierarchical models can only label the data they were fitted on.

In [ ]:
new_customers = pd.DataFrame({"Annual Income (k$)": [20, 90, 55], "Spending Score (1-100)": [85, 15, 50]})
mall_model = mall_cl.get_model(best_mall)
new_customers.assign(Cluster=mall_model.predict(new_customers))

## A7. Automatic checks: Mall Customers

In [ ]:
lb = mall_cl.leaderboard_
checks = {
    "k-scan picked 5 clusters":                      mall_cl.ctx.k == 5,
    "10+ models compared":                           len(lb) >= 10,
    "best model found 5 clusters with no noise":     mall_cl._score(mall_cl.labels(best_mall))["Clusters"] == 5
                                                     and mall_cl._score(mall_cl.labels(best_mall))["Noise %"] == 0,
    "best silhouette is above 0.5":                  mall_cl.tuned_.iloc[0, 1] > 0.5,
    "ignored columns kept in assign_clusters()":     {"CustomerID", "Gender", "Age", "Cluster"} <= set(seg.columns),
    "profile sizes add up to 200":                   mall_cl.profile(best_mall)["Size"].sum() == 200,
    "every model has recommended params":            all(m in mall_cl.recommended_params_ for m in mall_cl.tuned_.index),
    "new customers get a cluster label":             len(mall_model.predict(new_customers)) == 3,
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some Mall checks failed - see above"
print("\nAll Mall checks passed")

## A8. Chart checks: Mall Customers

In [ ]:
from matplotlib.axes import Axes
from matplotlib.figure import Figure

def run_chart_checks(calls, expect_errors=None):
    """Call every chart; PASS if it returns a matplotlib Axes/Figure without an error.
    expect_errors: {name: (callable, ExceptionType)} that SHOULD raise that error."""
    results = {}
    for name, call in calls.items():
        try:
            results[name] = isinstance(call(), (Axes, Figure))
        except Exception as e:
            print(f"  {name} -> {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, (call, err) in (expect_errors or {}).items():
        try:
            call()
            results[name] = False
        except err:
            results[name] = True
        except Exception as e:
            print(f"  {name} -> unexpected {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, ok in results.items():
        print(("PASS  " if ok else "FAIL  ") + name)
    return results


In [ ]:
calls = {
    "plot_k_scan()":                           lambda: mall_cl.plot_k_scan(),
    "plot_leaderboard()":                      lambda: mall_cl.plot_leaderboard(),
    "plot_leaderboard(metric='Davies-Bouldin')": lambda: mall_cl.plot_leaderboard(metric="Davies-Bouldin"),
    "plot_tuning()":                           lambda: mall_cl.plot_tuning(),
    "plot_clusters() - original columns":      lambda: mall_cl.plot_clusters(),
    "plot_clusters('dbscan') - with noise":    lambda: mall_cl.plot_clusters("dbscan"),
    "plot_silhouette()":                       lambda: mall_cl.plot_silhouette(),
    "plot_silhouette('dbscan')":               lambda: mall_cl.plot_silhouette("dbscan"),
    "plot_profile()":                          lambda: mall_cl.plot_profile(),
    "plot_profile('dbscan') - noise row":      lambda: mall_cl.plot_profile("dbscan"),
}
expect = {
    "plot_crosstab() without ground_truth raises": (lambda: mall_cl.plot_crosstab(), ValueError),
    "plot_tuning() before tune_top() raises":      (lambda: ClusterPicker(mall, ignore=["CustomerID", "Gender", "Age"]).plot_tuning(), RuntimeError),
}
chart_results = run_chart_checks(calls, expect)
assert all(chart_results.values()), "Some Mall chart checks failed - see above"
print("\nAll Mall chart checks passed")

# Part B: Palmer Penguins (do the clusters match the species?)
## B1. Load and clean
We cluster on the four body measurements only. `island` and `sex` are ignored, and `species` is passed as `ground_truth`: it's **never used for clustering**, only to score the result afterwards.

In [ ]:
PENGUIN_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(PENGUIN_URL).dropna().reset_index(drop=True)
print("Rows:", len(penguins))

peng_cl = ClusterPicker(penguins, ground_truth="species", ignore=["island", "sex"])
peng_cl.k_scan()

In [ ]:
peng_cl.plot_k_scan();           # silhouette, CH and DB say 2 - only BIC says 3

Notice the disagreement: **silhouette, Calinski-Harabasz and Davies-Bouldin all prefer k = 2, but GMM BIC prefers k = 3**.
Gentoo penguins are very different from the other two species, while Adelie and Chinstrap overlap. So the "roundest" split is 2 groups, even though there are 3 species.

In [ ]:
peng_lb = peng_cl.compare_models(verbose=False)
peng_lb[["Model", "Clusters", "Noise %", "Silhouette", "Mean rank", "ARI", "NMI", "Note"]]

## B2. Use domain knowledge: set k = 3
If you know there should be 3 groups, set `n_clusters=3`. Sorting by **ARI** (agreement with the true species; 1 = perfect, 0 = random) shows which model recovers them best.
In a real project you wouldn't have ground truth. This is a teaching exercise that shows the internal metrics alone can miss the real structure.

In [ ]:
peng3 = ClusterPicker(penguins, ground_truth="species", ignore=["island", "sex"], n_clusters=3, sort="ARI")
peng3_lb = peng3.compare_models(verbose=False)
peng3_lb[["Model", "Clusters", "Silhouette", "ARI", "NMI", "Homogeneity"]]

In [ ]:
peng3.tune_top(n=3, n_iter=20, verbose=False)
best_peng = peng3.best_model_id()
print(peng3.summary())
peng3.profile(best_peng)

In [ ]:
# How the clusters line up with the real species
pd.crosstab(penguins["species"], peng3.labels(best_peng), rownames=["species"], colnames=["cluster"])

In [ ]:
peng3.plot_clusters();           # 4 measurements, so this is a 2-D PCA view

In [ ]:
peng3.plot_crosstab();           # clusters vs the real species

In [ ]:
peng3.plot_profile();

## B3. Automatic checks: penguins

In [ ]:
checks = {
    "silhouette scan prefers k = 2":                   peng_cl.ctx.k == 2,
    "GMM BIC prefers k = 3":                           peng_cl.k_scan()["GMM BIC"].idxmin() == 3,
    "ground truth column not used as a feature":       "species" not in peng_cl.X.columns,
    "ARI/NMI/Homogeneity columns present":             {"ARI", "NMI", "Homogeneity"} <= set(peng_lb.columns),
    "with k = 3, best ARI is above 0.85":              peng3_lb["ARI"].max() > 0.85,
    "k = 3 beats k = 2 on ARI":                        peng3_lb["ARI"].max() > peng_lb["ARI"].max(),
    "profile shows the most common species":           "Most common label" in peng3.profile(best_peng).columns,
}

# Error handling
try:
    ClusterPicker(pd.read_csv(PENGUIN_URL), ignore=["species"])      # raw data has NaNs
    checks["missing values are rejected"] = False
except ValueError:
    checks["missing values are rejected"] = True
try:
    ClusterPicker(penguins.drop(columns="species"), sort="ARI")      # ARI needs ground truth
    checks["ARI sort without ground_truth is rejected"] = False
except ValueError:
    checks["ARI sort without ground_truth is rejected"] = True

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some penguin checks failed - see above"
print("\nAll checks passed")

## B4. Chart checks: penguins

In [ ]:
calls = {
    "plot_k_scan()":                      lambda: peng_cl.plot_k_scan(),
    "plot_leaderboard(metric='ARI')":     lambda: peng3.plot_leaderboard(metric="ARI"),
    "plot_tuning()":                      lambda: peng3.plot_tuning(),
    "plot_clusters() - PCA view":         lambda: peng3.plot_clusters(),
    "plot_silhouette()":                  lambda: peng3.plot_silhouette(),
    "plot_profile()":                     lambda: peng3.plot_profile(),
    "plot_crosstab()":                    lambda: peng3.plot_crosstab(),
    "plot_crosstab('kmeans')":            lambda: peng3.plot_crosstab("kmeans"),
}
chart_results = run_chart_checks(calls)
assert all(chart_results.values()), "Some penguin chart checks failed - see above"
print("\nAll checks passed")